# Week 56, Day 6: Forge: Week 56 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w056/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
from scipy.stats import norm
print("1.", round(np.exp(0.7), 3))
print("2.", 1 / (1 + 3))
print("3.", round(norm.cdf(2 / np.sqrt(2)), 3))

### Exercise: Part A: accuracy and calibration

Write ece(P, y) with 10 equal-width confidence bins. The three fitting functions are provided; compute each decoder's test probabilities, then store (accuracy, ECE) in results.

In [ ]:
import numpy as np
from scipy.optimize import minimize
r = np.random.default_rng(566)
C, N = 6, 60
pref = r.normal(size=(C, N)) * 0.2

def trials(n):
    y = r.integers(0, C, n)
    gain = np.exp(0.4 * r.normal(size=(n, 1)) - 0.08)
    return r.poisson(3.0 * np.exp(pref[y]) * gain).astype(float), y

X, y = trials(300); X_te, y_te = trials(3000)
mu, sd = X.mean(0), X.std(0) + 1e-9
Z, Z_te = (X - mu) / sd, (X_te - mu) / sd

def softmax(S):
    S = S - S.max(1, keepdims=True); E = np.exp(S); return E / E.sum(1, keepdims=True)

def fit_softmax(Z, y, lam=0.01):
    A = np.c_[np.ones(len(y)), Z]; Y = np.eye(C)[y]
    def f(w):
        W = w.reshape(A.shape[1], C); P = softmax(A @ W)
        G = A.T @ (P - Y) / len(y); G[1:] += lam * W[1:]
        return -np.mean(np.log(P[np.arange(len(y)), y])) + 0.5 * lam * np.sum(W[1:] ** 2), G.ravel()
    W = minimize(f, np.zeros(A.shape[1] * C), jac=True, method="L-BFGS-B").x.reshape(A.shape[1], C)
    return lambda Zn: softmax(np.c_[np.ones(len(Zn)), Zn] @ W)

def fit_lda(Z, y, gamma=0.3):
    mus = np.array([Z[y == k].mean(0) for k in range(C)]); Zc = Z - mus[y]
    S = Zc.T @ Zc / (len(y) - C); S = (1 - gamma) * S + gamma * np.trace(S) / Z.shape[1] * np.eye(Z.shape[1])
    W = np.linalg.solve(S, mus.T); b = -0.5 * np.sum(W * mus.T, 0)
    return lambda Zn: softmax(Zn @ W + b)

def fit_nb(X, y):
    rates = np.array([X[y == k].mean(0) for k in range(C)]) + 1e-3
    return lambda Xn: softmax(Xn @ np.log(rates).T - rates.sum(1))

def ece(P, y, bins=10):
    conf, pred = P.max(1), P.argmax(1)
    edges = np.linspace(0, 1, bins + 1); total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            total += ...
    return total

probs = {"softmax": fit_softmax(Z, y)(Z_te), "shrinkage LDA": fit_lda(Z, y)(Z_te), "naive Bayes": fit_nb(X, y)(X_te)}
results = {}
for name, P in probs.items():
    results[name] = ...
    print(f"{name:14s} accuracy {results[name][0]:.3f}, mean confidence {P.max(1).mean():.3f}, ECE {results[name][1]:.3f}")

In [ ]:
import numpy as np
def test_ece():
    "ece is zero for a perfectly calibrated toy example"
    P = np.array([[0.8, 0.2]] * 10); y_ = np.array([0] * 8 + [1] * 2)
    assert abs(ece(P, y_)) < 1e-9
def test_results():
    "results has accuracy and ECE for the three decoders, with accuracies above chance"
    assert set(results) == {"softmax", "shrinkage LDA", "naive Bayes"} and all(v[0] > 0.3 for v in results.values())
def test_nb():
    "naive Bayes is the worst calibrated of the three"
    assert results["naive Bayes"][1] == max(v[1] for v in results.values())

test_ece()
test_results()
test_nb()
print("All checks passed")

### Exercise: Part B: a threshold chosen by cost

Draw 300 validation trials. Compute the cost (1 per false alarm, 5 per miss) on a grid of thresholds for the softmax probability of category 0, choose the best, and evaluate it, the default 0.5 and the Bayes threshold 1/6 on the test set (cost per trial).

In [ ]:
dec = fit_softmax(Z, y)
X_va, y_va = trials(300); P_va = dec((X_va - mu) / sd)[:, 0]; P_te0 = probs["softmax"][:, 0]
t_va, t_te = (y_va == 0), (y_te == 0)
cost = lambda p, t, thr: (np.sum((p > thr) & ~t) + 5 * np.sum((p <= thr) & t)) / len(t)
grid = np.linspace(0.02, 0.9, 45)
thr_val = ...
cost_val_thr, cost_default, cost_bayes = cost(P_te0, t_te, thr_val), cost(P_te0, t_te, 0.5), cost(P_te0, t_te, 1 / 6)
print(f"validated threshold {thr_val:.2f}: test cost {cost_val_thr:.3f} per trial")
print(f"default 0.5: {cost_default:.3f}; Bayes 1/6: {cost_bayes:.3f}")

In [ ]:
import numpy as np
def test_costs():
    "the validated threshold beats the default 0.5 on test cost"
    assert cost_val_thr < cost_default
def test_near_bayes():
    "the validated threshold is within 0.15 of the Bayes threshold"
    assert abs(thr_val - 1 / 6) < 0.15

test_costs()
test_near_bayes()
print("All checks passed")